# Exploring the BFS federal vote JSON

**Goal:** understand how the Federal Statistical Office's vote results are structured so I can flatten every voting Sunday into one table.

**Data:** results for February 9th 2014, downloaded with `src/download.py`.

Source: BFS real-time vote data on opendata.swiss.

## 1. Load the file

The file is plain JSON, so Python's built-in `json` module is enough to read it. I use `utf-8-sig` encoding so that accented municipality names (e.g. Genève) load correctly.

In [1]:
import json
from pathlib import Path
import pandas as pd

path = Path("../data/raw/votes/20140209.json")
data = json.loads(path.read_text(encoding="utf-8-sig")) # loads accents correctly
data.keys()

dict_keys(['abstimmtag', 'timestamp', 'spatial_reference', 'schweiz'])

In [2]:
data["spatial_reference"]

[{'spatial_unit': 'kant', 'spatial_date': '1979-01-01'},
 {'spatial_unit': 'bezk', 'spatial_date': '2026-01-01'},
 {'spatial_unit': 'voge', 'spatial_date': '2026-01-01'}]

**Finding:** even though this vote took place in 2014, the results use the 2026 municipality list. Municipalities that merged since then are already combined, so I won't need to fix mergers by hand when joining other datasets.

## 2. Which votes were held that day?
Each voting Sunday can have several votes (`vorlagen`). Titles are stored in
five languages. I pull out the English one to keep the analysis readable.

In [3]:
def english_title(vorlage):
    return next(t["text"] for t in vorlage["vorlagenTitel"] if t["langKey"] == "en")

vorlagen = data["schweiz"]["vorlagen"]
for v in vorlagen:
    print(v["vorlagenId"], v["vorlageAngenommen"],
          round(v["resultat"]["jaStimmenInProzent"], 1), english_title(v))

5780 True 62.0 Federal decree on the financing and expansion of the railway infrastructure (direct counterproposal to the popular initiative 'For public transport')
5790 False 30.2 Popular initiative ‘Abortion funding is a private matter – relief for health insurance by removing the costs of abortion from mandatory basic insurance’
5800 True 50.3 Popular initiative ‘Against mass immigration’


**Finding:** 3 votes on 9 Feb 2014:
- Railway infrastructure funding (id 5780): accepted, ~62% yes
- Abortion financing initiative: rejected, ~30.2% yes
- Mass immigration initiative: accepted, ~50.3% yes

The immigration initiative is a well-known example of the French–German divide, so I use it for the rest of this notebook.

## 3. Structure
Path to municipality results:

`schweiz → vorlagen[] → kantone[] → gemeinden[] → resultat`

- 26 cantons per vote
- each municipality has a BFS number (`geoLevelnummer`), which is the key I'll use to join language region, demographics and map boundaries
- `bezirke` (districts) and `zaehlkreise` (city sub-districts) aren't needed

In [4]:
vote = vorlagen[0]
print(len(vote["kantone"]))
canton = vote["kantone"][0]
print(canton["geoLevelname"], canton.keys())
canton["gemeinden"][0]

26
Zürich dict_keys(['geoLevelnummer', 'geoLevelname', 'resultat', 'bezirke', 'gemeinden', 'zaehlkreise'])


{'geoLevelnummer': '1',
 'geoLevelname': 'Aeugst am Albis',
 'geoLevelParentnummer': '101',
 'resultat': {'gebietAusgezaehlt': True,
  'jaStimmenInProzent': 60.756501182,
  'jaStimmenAbsolut': 514,
  'neinStimmenAbsolut': 332,
  'stimmbeteiligungInProzent': 61.935953421,
  'eingelegteStimmzettel': 851,
  'anzahlStimmberechtigte': 1374,
  'gueltigeStimmen': 846}}

## 4. Flattening one vote into a table
**Result:** 2114 rows (one per municipality) and 11 columns.

**First look at the Röstigraben:** the 10 municipalities with the lowest yes-share are mostly in cantons Vaud, Fribourg and Jura.

In [5]:
immigration = next(v for v in vorlagen if "immigration" in english_title(v).lower())

rows = []
for canton in immigration["kantone"]:
    for g in canton["gemeinden"]:
        rows.append({
            "canton": canton["geoLevelname"],
            "bfs_nr": int(g["geoLevelnummer"]),
            "municipality": g["geoLevelname"],
            **g["resultat"], # copy every number from resultat into the row
        })
        
df = pd.DataFrame(rows)
print(df.shape)
df.sort_values("jaStimmenInProzent").head(10)

(2114, 11)


,canton,bfs_nr,municipality,gebietAusgezaehlt,jaStimmenInProzent,jaStimmenAbsolut,neinStimmenAbsolut,stimmbeteiligungInProzent,eingelegteStimmzettel,anzahlStimmberechtigte,gueltigeStimmen
1705,Vaud,5640,Lussy-sur-Morges,True,19.047619,48,204,63.184080,254,402,252
2074,Jura,6716,Mettembert,True,21.818182,12,43,68.750000,55,80,55
1607,Vaud,5476,Chevilly,True,24.468085,23,71,59.281437,99,167,94
1776,Vaud,5750,Les Clées,True,24.719101,22,67,80.000000,92,115,89
1656,Vaud,5555,Corcelles-près-Concise,True,26.428571,37,103,62.882096,144,229,140
1756,Vaud,5722,Grens,True,26.573427,38,105,70.000000,147,210,143
1712,Vaud,5650,Vaux-sur-Morges,True,26.666667,24,66,79.310345,92,116,90
722,Fribourg / Freiburg,2230,Villarsel-sur-Marly,True,26.666667,12,33,83.333333,45,54,45
1686,Vaud,5610,Saint-Saphorin (Lavaux),True,26.829268,44,120,73.390558,171,233,164
1826,Vaud,5863,Vinzel,True,27.200000,34,91,64.878049,133,205,125


## 5. Sanity check

The difference is 0 votes (0%).

In [6]:
print("Sum of municipalities:", df["jaStimmenAbsolut"].sum())
print("Official national total:", immigration["resultat"]["jaStimmenAbsolut"])

Sum of municipalities: 1463854
Official national total: 1463854


## Summary and next steps
- Municipality-level results are easy to reach: two nested loops.
- The data already uses current municipality boundaries.
- Next: loop over every voting Sunday since 2010, flatten all votes into one long table, and save it as Parquet.